# NFL Salary Cap Allocation & Team Performance

**Question:** Every NFL team has the same salary budget (the cap). Does how a team divides that budget across positions, and whether it pays one star or spreads money across a group, relate to how well it performs?

This notebook runs top to bottom. See `PRD.md` for the full spec and `README.md` for the write-up.

## §0 Setup and constants

We load the libraries and set up a few values that every later section uses: the seasons we study, the league salary cap for each year, and a fix for teams that moved cities.

*Summary:* No data is touched yet. We're just writing down the fixed facts in one place, so if one is wrong we only have to change it here.

In [1]:
import os
import nflreadpy as nfl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Git does not store empty folders, so create them here for a fresh clone
for folder in ["data/raw", "data/processed", "charts"]:
    os.makedirs(folder, exist_ok=True)

# range() stops before the end number, so 2026 means "through 2025"
SEASONS = list(range(2013, 2026))

# League salary cap per season, in millions of dollars (source: NFL / OverTheCap).
# Hardcoded because it is only 13 numbers, and they never change once a season is set.
# Note the 2021 drop: the cap shrank because of COVID revenue losses.
LEAGUE_CAP = {
    2013: 123.0, 2014: 133.0, 2015: 143.28, 2016: 155.27, 2017: 167.0,
    2018: 177.2, 2019: 188.2, 2020: 198.2, 2021: 182.5, 2022: 208.2,
    2023: 224.8, 2024: 255.4, 2025: 279.2,
}

# Three teams moved cities during our window. We always use the team's CURRENT
# abbreviation, so a franchise stays one team across all 13 seasons.
TEAM_FIX = {
    "OAK": "LV",   # Oakland Raiders -> Las Vegas (2020)
    "SD": "LAC",   # San Diego Chargers -> Los Angeles (2017)
    "STL": "LA",   # St. Louis Rams -> Los Angeles (2016)
}


def standardize_team(abbreviation):
    """Return the current abbreviation for a team (unchanged if it never moved)."""
    return TEAM_FIX.get(abbreviation, abbreviation)

## §1 Results: point differential per game

We download every game from 2013–2025, keep only regular-season games, and turn each game into two rows: one for the home team and one for the away team. Then we add up each team's season.

*Summary:* The output is one row per team per season with wins, points scored and points allowed. Our main outcome is **point differential per game** (points scored minus points allowed, divided by games). We divide by games because the season grew from 16 to 17 games in 2021.

In [2]:
# nflreadpy returns a Polars table, so convert to pandas right away
schedules = nfl.load_schedules(SEASONS).to_pandas()

# Save an untouched copy so we can see exactly what we downloaded
schedules.to_csv("data/raw/schedules.csv", index=False)

# Keep regular season only (playoffs would reward good teams with extra games)
reg_season = schedules[schedules["game_type"] == "REG"]

# Drop games with no score. The 2022 Bills-Bengals game was cancelled,
# so both of those teams played 16 games instead of 17 that year.
reg_season = reg_season.dropna(subset=["home_score", "away_score"])

print("Regular-season games:", len(reg_season))

Regular-season games: 3407


In [3]:
# Each game has a home team and an away team. We want one row per TEAM per game,
# so we build a home-team view and an away-team view, then stack them.
home_games = pd.DataFrame({
    "season": reg_season["season"],
    "team": reg_season["home_team"],
    "points_for": reg_season["home_score"],
    "points_against": reg_season["away_score"],
})
away_games = pd.DataFrame({
    "season": reg_season["season"],
    "team": reg_season["away_team"],
    "points_for": reg_season["away_score"],
    "points_against": reg_season["home_score"],
})
team_games = pd.concat([home_games, away_games], ignore_index=True)

# Relocated teams get their current abbreviation
team_games["team"] = team_games["team"].apply(standardize_team)

# Win = 1, tie = 0.5, loss = 0 (the standard way NFL win % counts ties)
team_games["win"] = 0.0
team_games.loc[team_games["points_for"] > team_games["points_against"], "win"] = 1.0
team_games.loc[team_games["points_for"] == team_games["points_against"], "win"] = 0.5

team_games.head()

,season,team,points_for,points_against,win
0,2013,DEN,49,27,1.0
1,2013,BUF,21,23,0.0
2,2013,CAR,7,12,0.0
3,2013,CHI,24,21,1.0
4,2013,CLE,10,23,0.0


In [4]:
# Add up each team's season
results = team_games.groupby(["team", "season"]).agg(
    games=("win", "count"),
    wins=("win", "sum"),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum"),
).reset_index()

# Per-game versions, so 16- and 17-game seasons are comparable
results["win_pct"] = results["wins"] / results["games"]
results["pf_per_game"] = results["points_for"] / results["games"]
results["pa_per_game"] = results["points_against"] / results["games"]
results["point_diff_per_game"] = results["pf_per_game"] - results["pa_per_game"]

# Checks: if any of these fail, something upstream is wrong
assert len(results) == 32 * 13, f"Expected 416 team-seasons, got {len(results)}"
assert results["team"].nunique() == 32, "Relocated teams were not merged"
assert results["games"].isin([16, 17]).all(), "A team-season has an odd number of games"

results.describe().round(2)

,season,games,wins,points_for,points_against,win_pct,pf_per_game,pa_per_game,point_diff_per_game
count,416.00,416.00,416.00,416.00,416.00,416.00,416.00,416.00,416.00
mean,2019.00,16.38,8.19,373.85,373.85,0.50,22.83,22.83,0.00
std,3.75,0.49,3.14,69.96,55.77,0.19,4.27,3.40,6.07
min,2013.00,16.00,0.00,224.00,225.00,0.00,13.88,14.06,-13.38
25%,2016.00,16.00,6.00,322.75,332.00,0.38,19.86,20.34,-4.50
50%,2019.00,16.00,8.00,369.00,372.50,0.50,22.56,22.59,-0.24
75%,2022.00,17.00,11.00,420.00,413.00,0.65,25.96,25.06,4.51
max,2025.00,17.00,15.00,606.00,534.00,0.94,37.88,32.44,15.56


In [5]:
# Sanity check against a known season: the 2023 Chiefs scored 371 and allowed 294
# in 17 games, so point differential per game should be (371 - 294) / 17 = 4.53
results[(results["team"] == "KC") & (results["season"] == 2023)]

,team,season,games,wins,points_for,points_against,win_pct,pf_per_game,pa_per_game,point_diff_per_game
205,KC,2023,17,11.0,371,294,0.647059,21.823529,17.294118,4.529412
